# 🏗️ Aula 03 — Engenharia de Features para um Reator CSTR

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Dataset:** CSTR de etoxilação — dados de processo (1/min) + análises de laboratório (1/10min)

---

## Contexto

Você é o engenheiro responsável por um **reator CSTR de etoxilação**. O sensor de temperatura e vazão estão em tempo real (1 leitura/min), mas a composição de saída (conversão) só é analisada no laboratório a cada 10 min. Seu objetivo: criar features para que um modelo de ML possa **estimar a conversão em tempo real** (soft-sensor).

## 3.1 — Exercício Guiado: Criando Features do Zero

Siga passo-a-passo as células abaixo.

### Passo 1: Importar bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Passo 2: Carregar os dois datasets

In [ ]:
BASE = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula03/"

df_proc = pd.read_csv(BASE + 'cstr_processo_1min.csv', parse_dates=['timestamp'])
df_lab  = pd.read_csv(BASE + 'cstr_lab_10min.csv', parse_dates=['timestamp'])

print(f"Processo: {df_proc.shape[0]} linhas (1/min)")
print(f"Lab:      {df_lab.shape[0]} linhas (1/10min)")

### Passo 3: Alinhar frequências (resample para 10 min)

In [ ]:
df_proc.set_index('timestamp', inplace=True)
df_lab.set_index('timestamp', inplace=True)

# Resample: média de 10 leituras de sensor = 1 valor alinhado com o lab
df_proc_10min = df_proc.resample('10min').mean()

# Juntar com target
df = df_proc_10min.join(df_lab, how='inner')
print(f"Shape após resample + join: {df.shape}")
print(f"Linhas perdidas por join: {len(df_proc_10min) - len(df)}")

### Passo 4: Criar lags de T_reator

In [ ]:
# Cada lag de 1 passo = 10 min atrás (porque resampleamos para 10 min)
df['T_reator_C_lag1'] = df['T_reator_C'].shift(1)   # 10 min atrás
df['T_reator_C_lag2'] = df['T_reator_C'].shift(2)   # 20 min atrás
df['T_reator_C_lag3'] = df['T_reator_C'].shift(3)   # 30 min atrás

### Passo 5: Criar média móvel e taxa de variação de T_reator

In [ ]:
df['T_reator_C_ma3'] = df['T_reator_C'].rolling(3).mean()   # média dos últimos 30 min
df['dT_reator']      = df['T_reator_C'].diff()               # derivada discreta (°C/10min)

### Passo 6: Repetir para vazão de alimentação

In [ ]:
df['vazao_L_min_lag1'] = df['vazao_L_min'].shift(1)
df['vazao_L_min_lag2'] = df['vazao_L_min'].shift(2)
df['vazao_L_min_ma3']  = df['vazao_L_min'].rolling(3).mean()
df['dvazao']           = df['vazao_L_min'].diff()

### Passo 7: Limpar e verificar

In [ ]:
df.dropna(inplace=True)
print(f"Shape final: {df.shape}")
print(f"Features: {[c for c in df.columns if c != 'conversao']}")
print(f"Target: conversao")

### ✏️ Pausa reflexiva (2 min)

Qual lag você acha que será **mais importante** para predizer a conversão? Por quê?

> _Escreva aqui..._

---

## 3.2 — Exercício em Grupo: Selecionando as Melhores Features

Com o dataset completo de features construído:

In [ ]:
corr = df.corr()['conversao'].sort_values(ascending=False)
print("Correlação de cada feature com a conversão:")
print(corr)
print(f"\nTop 5 features:")
print(corr.head(5))

### Análise

Para cada uma das 5 features mais correlacionadas:
1. **Por que a correlação faz sentido fisicamente?**
2. **Há alguma feature que você esperava no top-5 e não está? Por quê?**

> _Escreva aqui suas análises..._

### 🧠 Desafio extra (NT)

A feature `dT_reator` (derivada) apareceu onde? Ela é útil mesmo se a correlação linear for baixa? Por quê?

> _Escreva aqui..._

---

## Checklist de Validação

- [ ] Alinhamento temporal OK (resample + join)
- [ ] Lags criados (≥2 por variável)
- [ ] Médias móveis criadas (≥1 por variável)
- [ ] Taxa de variação criada (≥1 por variável)
- [ ] Matriz de correlação calculada
- [ ] Top-5 features identificadas
- [ ] 1 parágrafo de interpretação física